# Module 1: Foundations

Build a minimal Strands agent, run it, and inspect every step of the **agentic loop** in action.

The agentic loop is the core of every Strands agent:

1. **Input & Context**: the user prompt enters the loop
2. **Reasoning (LLM)**: the model decides what to do next
3. **Tool Selection**: if it needs data, the model picks a tool
4. **Tool Execution**: your Python function runs, feeding its result back into step 2
5. **Response**: the loop exits once the model has enough information

You write the tools. Strands runs the loop.

**Prerequisites:** Python 3.10+, AWS credentials with Amazon Bedrock access.

---

## Step-1: Install requirements

In [ ]:
%pip install -r requirements.txt

---

## AI Agent Primitives

An agent = a model given tools, instructions, and memory, running in a loop until the goal is met.

| Primitive | What it is | In Strands |
|-----------|-----------|-----------|
| **Model (LLM)** | The reasoning engine: plans, decides, responds | `model=` on `Agent` (defaults to Claude Sonnet 4.6 on Amazon Bedrock) |
| **Tools** | Python functions the model can call to act | `@tool` decorated functions |
| **Prompt & Instructions** | Role, goals, and guardrails that steer behavior | `system_prompt=` parameter on `Agent` |
| **Memory & Context** | Conversation history and state across turns | `agent.messages` list, grows each turn |

These four primitives are the building blocks. Every multi-agent pattern in this workshop is built from combinations of them.

---

## Step-2: Create and Run an Agent

The smallest agent is a model plus a system prompt. With no tools, the loop runs once: the model answers from what it already knows.

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────
# No model= is passed, so every Agent uses the Strands default on Amazon Bedrock:
# Claude Sonnet 4.6 through the global inference profile (global.anthropic.claude-sonnet-4-6).
# To use another model enabled in your account, create one and pass model=model to each Agent:
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")  # faster
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")                        # Amazon Nova

from strands import Agent

agent = Agent(system_prompt="You are a Math specialist.")

result = agent("What is the 3rd number in a Fibonacci series? Show the intermediate steps.")

---

## Step-3: Define a Tool

Tools are plain Python functions decorated with `@tool`. The model reads the **docstring** to decide when and how to call them. The docstring is the routing logic, not code.

Write docstrings for the model, not for other developers.

In [ ]:
from strands import tool

@tool
def get_market_data(segment: str) -> str:
    """Get subscription market data for a customer segment.

    Args:
        segment: The customer segment to look up (e.g. 'premium', 'standard', 'enterprise')
    """
    data = {
        "premium": {"willingness_to_pay": "68%", "avg_monthly": "$22", "churn_rate": "8%"},
        "standard": {"willingness_to_pay": "41%", "avg_monthly": "$14", "churn_rate": "18%"},
        "enterprise": {"willingness_to_pay": "85%", "avg_monthly": "$45", "churn_rate": "4%"},
    }
    row = data.get(segment.lower(), {"error": f"no data for segment '{segment}'"})
    return str(row)

print("Tool defined. Docstring the model will read:")
print(get_market_data.__doc__)

---

## Step-4: Give the Agent the Tool

Wire the tool into an `Agent` with a system prompt and call it. This time the loop runs more than once: the model calls the tool, reads the result, then answers.

In [ ]:
agent = Agent(
    tools=[get_market_data],
    system_prompt="You are a market research assistant. Use your tools to answer questions accurately.",
)

result = agent("What percentage of premium segment customers are willing to pay for a subscription tier?")

---

## Step-5: Inspect the Agent Loop

Every step is stored in `agent.messages`. The model's tool calls are `toolUse` blocks in `assistant` messages; the tool results come back as `toolResult` blocks in `user` messages.

In [ ]:
import json

print(f"Total messages: {len(agent.messages)}")
print("=" * 60)

for i, msg in enumerate(agent.messages):
    for block in msg["content"]:
        if msg["role"] == "user" and "text" in block:
            print(f"[{i}] USER:         {block['text'][:80]}")
        elif "toolResult" in block:
            for item in block["toolResult"]["content"]:
                if "text" in item:
                    print(f"[{i}] TOOL RESULT:  {item['text'][:80]}")
        elif "toolUse" in block:
            tu = block["toolUse"]
            print(f"[{i}] TOOL CALL:    {tu['name']}({json.dumps(tu['input'])})")
        elif "text" in block:
            print(f"[{i}] LLM RESPONSE: {block['text'][:80]}")

---

## Step-6: Metrics

Every `Agent.__call__()` returns an `AgentResult`. `result.metrics.get_summary()` gives you token usage, loop cycles and per-tool stats with no extra setup.

In [ ]:
summary = result.metrics.get_summary()
usage = summary["accumulated_usage"]
print("Loop metrics:")
print(f"  LLM calls (cycles):  {summary['total_cycles']}")
print(f"  Input tokens:        {usage['inputTokens']}")
print(f"  Output tokens:       {usage['outputTokens']}")
print(f"  Total tokens:        {usage['totalTokens']}")
for name, data in summary["tool_usage"].items():
    print(f"  Tool {name}: calls={data['execution_stats']['call_count']}")

---

## Step-7: Multi-turn Conversation

The agent keeps its conversation history across calls. Each `agent(...)` call adds to the same context window, so the agent remembers what happened in previous turns.

In [ ]:
chat_agent = Agent(
    tools=[get_market_data],
    system_prompt="You are a market research assistant. Use your tools to answer questions accurately.",
)

chat_agent("What is the willingness to pay for the premium segment?")
chat_agent("And for the standard segment?")
chat_agent("Which segment has the lower churn rate between those two?")

print(f"\nConversation length: {len(chat_agent.messages)} messages")

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| `@tool` | Python function the model can call; docstring is the routing logic |
| `Agent(tools=[], system_prompt=...)` | Assembles model + tools + instructions |
| `agent.messages` | Full loop history: user turns, LLM responses, tool calls, tool results |
| `result.metrics.get_summary()` | Token usage, cycle count and per-tool stats on every call |
| Multi-turn | Each `agent(...)` call extends the same context window |

---

## What's Next

In **Module 2: Single Agent**, you use these same building blocks with three business intelligence tools to build a Strategy Advisor for the MarketNest decision brief, and find where one agent hits its ceiling.